# 3. Predict and Evaluate

For each **test** user (users the model has never seen):

1. compute the likelihood of their actions under the **no-call** model;
2. compute the likelihood of their actions under the **call** model;
3. **argmax**: the user belongs to the group whose model gives the higher likelihood.

If the sequence fits the transition matrix of the callers best, the user is predicted to call,
and we call them first.

Then we check how good these predictions are, and compare them with simple baselines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("data")
MODELS_DIR = Path("models")

GROUPS = ["no call", "call"]  # group index 0 and 1, same as the `called` column

model = np.load(MODELS_DIR / "markov_models.npz")
true_process = np.load(DATA_DIR / "true_process.npz")
test = pd.read_csv(DATA_DIR / "test.csv")

ACTIONS = [str(a) for a in model["actions"]]
ACTION_INDEX = {action: i for i, action in enumerate(ACTIONS)}
test["sequence"] = test["actions"].map(lambda text: [ACTION_INDEX[a] for a in text.split()])

called = test["called"].to_numpy().astype(bool)
print(f"{len(test)} test users, {called.sum()} of them called ({called.mean():.1%})")

## 3.1 Log-likelihood

The likelihood of a sequence $x_1, \dots, x_n$ under the model of group $c$ is

$$L_c = \pi_c(x_1) \cdot A_c(x_1, x_2) \cdot A_c(x_2, x_3) \cdots A_c(x_{n-1}, x_n)$$

This is a product of many small numbers, so it quickly becomes too small for the computer
(*underflow*). We take the logarithm, which turns the product into a sum:

$$\log L_c = \log \pi_c(x_1) + \sum_{t=1}^{n-1} \log A_c(x_t, x_{t+1})$$

The logarithm is increasing, so the group with the highest $\log L_c$ is also the group with the highest $L_c$.
The argmax does not change.

In [ ]:
def log_likelihood(sequence, first_action, matrix):
    total = np.log(first_action[sequence[0]])
    for current, following in zip(sequence[:-1], sequence[1:]):
        total += np.log(matrix[current, following])
    return total

## 3.2 One user, step by step

We follow one test user through their sequence. After each step we show the log-likelihood
accumulated so far under each group, and which group is ahead.

The prediction is the group that is ahead **at the end of the sequence**.

In [ ]:
example = test[called & test["actions"].str.contains("fx_transfer_failed")].iloc[0]
sequence = example["sequence"]

rows = [{
    "step": f"start: {ACTIONS[sequence[0]]}",
    "log P (no call)": np.log(model["first_no_call"][sequence[0]]),
    "log P (call)": np.log(model["first_call"][sequence[0]]),
}]
for current, following in zip(sequence[:-1], sequence[1:]):
    rows.append({
        "step": f"{ACTIONS[current]} → {ACTIONS[following]}",
        "log P (no call)": np.log(model["matrix_no_call"][current, following]),
        "log P (call)": np.log(model["matrix_call"][current, following]),
    })

steps = pd.DataFrame(rows)
steps["log L so far (no call)"] = steps["log P (no call)"].cumsum()
steps["log L so far (call)"] = steps["log P (call)"].cumsum()
steps["group ahead"] = np.where(
    steps["log L so far (call)"] > steps["log L so far (no call)"], "call", "no call")

print(f"User {example['user_id']}, called = {example['called']}")
print(f"Prediction (argmax at the last step): {steps['group ahead'].iloc[-1]}")
steps.round(3)

## 3.3 Log-likelihood of every test user under each group

We put the two log-likelihoods side by side. Column 0 is the no-call group, column 1 is the call group,
so `argmax` over the columns directly gives the predicted group.

In [ ]:
log_lik = np.column_stack([
    test["sequence"].map(lambda s: log_likelihood(s, model["first_no_call"], model["matrix_no_call"])),
    test["sequence"].map(lambda s: log_likelihood(s, model["first_call"], model["matrix_call"])),
])
test["log_lik_no_call"] = log_lik[:, 0]
test["log_lik_call"] = log_lik[:, 1]
test["predicted"] = log_lik.argmax(axis=1)

test[["user_id", "n_actions", "log_lik_no_call", "log_lik_call", "predicted", "called"]].head().round(2)

## 3.4 Argmax, with and without the prior

**Maximum likelihood (ML)** — the method as described: the group whose model explains the sequence best.

$$\hat{c} = \arg\max_c \; \log L_c$$

**Maximum a posteriori (MAP)** — the same argmax, but each group also gets its prior probability
$P(c)$, i.e. how common the group is:

$$\hat{c} = \arg\max_c \; \big[\log L_c + \log P(c)\big]$$

Calling is rare ($P(\text{call}) \approx 0.05$), so with MAP the call group needs a much better fit
to win. ML treats both groups as equally likely in advance, so it predicts "call" for many more users.
It catches more callers but also calls many users who would not call.

In [ ]:
call_rate = float(model["call_rate"])
log_prior = np.log([1 - call_rate, call_rate])

test["predicted_ml"] = log_lik.argmax(axis=1)
test["predicted_map"] = (log_lik + log_prior).argmax(axis=1)

print(f"Predicted 'call' by ML:  {test['predicted_ml'].sum()} users")
print(f"Predicted 'call' by MAP: {test['predicted_map'].sum()} users")

## 3.5 Baselines

A model is useful only if it beats simple alternatives:

- **Never call anyone** — always predict "no call". Accuracy is high because callers are rare,
  but it catches no callers. This shows why **accuracy is a bad metric here**.
- **Simple rule** — predict "call" if the user had any failure or opened the help page.

Metrics:

- **precision** — of the users we call, how many really would have called;
- **recall** — of all users who called, how many we caught.

In [ ]:
rule_flag = test["actions"].str.contains("failed|open_help_page").to_numpy()


def evaluate(flagged):
    caught = int((flagged & called).sum())
    return {
        "users flagged": int(flagged.sum()),
        "callers caught": caught,
        "precision": caught / flagged.sum() if flagged.sum() else np.nan,
        "recall": caught / called.sum(),
        "accuracy": (flagged == called).mean(),
    }


results = pd.DataFrame({
    "never call": evaluate(np.zeros(len(test), dtype=bool)),
    "rule: failure or help": evaluate(rule_flag),
    "Markov argmax (ML)": evaluate(test["predicted_ml"].to_numpy() == 1),
    "Markov argmax with prior (MAP)": evaluate(test["predicted_map"].to_numpy() == 1),
}).T
results.round(3)

## 3.6 Picture of the argmax

Each point is one test user: x = log-likelihood under the no-call model, y = under the call model.

- above the diagonal → the call model fits better → argmax says **call**;
- below the diagonal → argmax says **no call**.

Red points are users who really called. Many of them are below the line: their short or ordinary
sequences fit the no-call model just as well. This is why the prediction cannot be perfect.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(test.loc[~called, "log_lik_no_call"], test.loc[~called, "log_lik_call"],
           s=4, alpha=0.3, color="steelblue", label="did not call")
ax.scatter(test.loc[called, "log_lik_no_call"], test.loc[called, "log_lik_call"],
           s=8, alpha=0.7, color="firebrick", label="called")
low, high = log_lik.min(), log_lik.max()
ax.plot([low, high], [low, high], color="black", linestyle="--", label="equal likelihood")
ax.set_xlabel("log L (no-call model)")
ax.set_ylabel("log L (call model)")
ax.legend()
plt.show()

## 3.7 From argmax to a probability

The argmax gives one group per user. The call center, however, can make only a limited number of
calls, so it also needs to know **which users to call first**.

The same two numbers give the posterior probability of each group (Bayes' rule):

$$P(c \mid \text{actions}) = \frac{L_c \, P(c)}{\sum_{k} L_k \, P(k)}$$

The argmax of this probability is exactly the MAP prediction. Sorting users by
$P(\text{call} \mid \text{actions})$ gives the order in which to call them.
We compute it in log space to avoid underflow.

In [ ]:
def posterior(log_lik, log_prior):
    joint = log_lik + log_prior
    joint = joint - joint.max(axis=1, keepdims=True)  # avoid overflow in exp
    probabilities = np.exp(joint)
    return probabilities / probabilities.sum(axis=1, keepdims=True)


test["p_call"] = posterior(log_lik, log_prior)[:, 1]
assert (test["p_call"].gt(0.5) == (test["predicted_map"] == 1)).all()

test.sort_values("p_call", ascending=False)[["user_id", "actions", "p_call", "called"]].head(10)

## 3.8 Ranking quality: ROC AUC

**ROC AUC** measures how well the probability orders the users: if we pick one random caller and one
random non-caller, AUC is the probability that the caller gets the higher $P(\text{call} \mid \text{actions})$.
0.5 = random, 1.0 = perfect.

We also compute the **oracle**: the same calculation, but with the **true** matrices that created the data.
No model can do better on average, so the oracle shows the best AUC this data allows.

In [ ]:
def roc_auc(is_positive, values):
    ranks = pd.Series(values).rank().to_numpy()
    n_pos = is_positive.sum()
    n_neg = len(is_positive) - n_pos
    return (ranks[is_positive].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


true_rate = float(true_process["call_rate"])
oracle_log_lik = np.column_stack([
    test["sequence"].map(lambda s: log_likelihood(s, true_process["first_action"], true_process["matrix_no_call"])),
    test["sequence"].map(lambda s: log_likelihood(s, true_process["first_action"], true_process["matrix_call"])),
])
test["oracle_p_call"] = posterior(oracle_log_lik, np.log([1 - true_rate, true_rate]))[:, 1]

p_call = test["p_call"].to_numpy()
oracle_p_call = test["oracle_p_call"].to_numpy()

pd.Series({
    "random": 0.5,
    "rule: failure or help": roc_auc(called, rule_flag.astype(float)),
    "Markov model": roc_auc(called, p_call),
    "oracle (true matrices)": roc_auc(called, oracle_p_call),
}, name="ROC AUC").round(3).to_frame()

## 3.9 The business view: a daily call budget

If the call center can call only K users, it calls the K users with the highest $P(\text{call} \mid \text{actions})$.
The practical question is: *how many of them are real callers?*

This is **precision at K**. Calling K random users gives precision equal to the call rate.

In [ ]:
def precision_at_k(values, k):
    top_k = np.argsort(-values)[:k]
    return called[top_k].mean()


budgets = [25, 50, 100, 200, 300, 500, 1000]
at_k = pd.DataFrame({
    "random": [called.mean()] * len(budgets),
    "Markov model": [precision_at_k(p_call, k) for k in budgets],
    "oracle": [precision_at_k(oracle_p_call, k) for k in budgets],
}, index=pd.Index(budgets, name="K (users called)"))
at_k["callers caught (model)"] = [int(called[np.argsort(-p_call)[:k]].sum()) for k in budgets]

ax = at_k[["random", "Markov model", "oracle"]].plot(marker="o", figsize=(8, 4))
ax.set_ylabel("precision at K")
ax.set_ylim(0, 1)
plt.show()

at_k.round(3)

## 3.10 Does the window length matter?

Users with few actions give the model little evidence: both models fit a short sequence almost
equally well. We compare the AUC for short and long windows.

In [ ]:
test["length_group"] = pd.cut(test["n_actions"], bins=[2, 6, 10, 15, 20],
                              labels=["3-6", "7-10", "11-15", "16-20"])

rows = []
for group, part in test.groupby("length_group", observed=True):
    is_caller = part["called"].to_numpy().astype(bool)
    rows.append({
        "actions": group,
        "users": len(part),
        "callers": int(is_caller.sum()),
        "AUC model": roc_auc(is_caller, part["p_call"].to_numpy()),
        "AUC oracle": roc_auc(is_caller, part["oracle_p_call"].to_numpy()),
    })
pd.DataFrame(rows).round(3)

## 3.11 Questions to discuss

1. How close is the Markov model to the oracle? The gap comes from estimation error,
   mostly in the small callers group.
2. Which decision fits the business: argmax (ML), argmax with the prior (MAP), or a fixed call budget (top K)?
3. The data here follow the model's assumptions exactly. What happens when they do not, for example:
   - callers behave normally and change only after a single failure (behavior changes over time);
   - the order of actions does not matter, only how many failures there were;
   - the session length itself is a signal?